In [ ]:
!pip install transformers datasets torchmetrics

In [ ]:
import torch
import numpy as np

from datasets import load_dataset, Audio, DatasetDict

from transformers import AutoModel, AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Χρήση συσκευής: {device}")

In [ ]:
model_or = AutoModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/AST-finetuned-Ravdess").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/AST-finetuned-Ravdess")

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

dataset = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

test_dataset = dataset["test"]
print(f"Μέγεθος test split: {len(test_dataset)} δείγματα")

In [ ]:
def extract_embeddings(model, feature_extractor, dataset, num_samples=150, encoder_attr=None):
    probe_model = getattr(model, encoder_attr) if encoder_attr else model
    num_layers = model.config.num_hidden_layers + 1
    layer_embeddings = [[] for _ in range(num_layers)]

    for i in range(num_samples):
        sample = dataset[i]
        audio = sample["audio"]["array"]
        inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = probe_model(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(outputs.hidden_states):
            pooled = h.mean(dim=1)
            layer_embeddings[layer].append(pooled.squeeze().cpu().numpy())

    return [np.array(layer) for layer in layer_embeddings]

In [ ]:
emb_pretrained = extract_embeddings(model_or, feature_extractor, test_dataset, encoder_attr=None)
emb_finetuned = extract_embeddings(model_ft, feature_extractor, test_dataset, encoder_attr="audio_spectrogram_transformer")

del model_or, model_ft
torch.cuda.empty_cache()

In [ ]:
#Συνάρτηση υπολογισμού του CKA
def linear_CKA(X, Y):

    X = X - X.mean(0)
    Y = Y - Y.mean(0)

    hsic = np.linalg.norm(X.T @ Y, ord="fro") ** 2

    var1 = np.linalg.norm(X.T @ X, ord="fro")
    var2 = np.linalg.norm(Y.T @ Y, ord="fro")

    return hsic / (var1 * var2)

In [ ]:
import pandas as pd

num_layers = len(emb_pretrained)
cka_scores = [linear_CKA(emb_pretrained[l], emb_finetuned[l]) for l in range(num_layers)]

cka_results = pd.DataFrame({"layer": range(num_layers), "cka": cka_scores})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cka_results["layer"], cka_results["cka"], marker="o", color="#065A82")
plt.xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
plt.ylabel("CKA similarity (Pretrained vs Fine-tuned)")
plt.title("Layer-wise CKA: AST πριν/μετά το Fine-tuning (RAVDESS)")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()